# Week 6 · Deep dive (optional): retrieval engineering and tools

**Optional and not graded.** Do the [lab](lab.ipynb) first: it ships `/v1/ask`. This notebook goes deeper into the [lecture notes](lecture.md). You will measure retrieval like a search engineer, build keyword, hybrid and multi-query search from scratch, test chunking and embedding sizes, push the tool-calling protocol to its edges, and try the two ways Agent Framework can do RAG, plus a platform tool.

| Part | What you explore | Lecture notes |
|---|---|---|
| 1 | Measure retrieval: recall@k and MRR, and why the public questions are too easy | §5, §7 |
| 2 | Keyword search from scratch (BM25) | §5 |
| 3 | Hybrid search: Reciprocal Rank Fusion and its weights | §5 |
| 4 | Query rewriting and multi-query fusion: the first step of agentic retrieval | §8, §12 |
| 5 | Chunking: sections versus fixed-size windows | §4 |
| 6 | Embedding size: 1,536, 256 and 64 dimensions | §2 |
| 7 | The tool-calling protocol at its edges: parallel calls, `tool_choice`, errors as data | §9, §10 |
| 8 | RAG as a tool versus RAG as a context provider (Agent Framework) | §12, §13 |
| 9 | A platform tool next to your own: Code Interpreter from Agent Framework | §11 |

Open it after you have submitted your URL: it does not edit `app/` or deploy, so it cannot change your score. Run the setup cell, then the parts in order. Part 1 builds the index, the question sets and the client that later parts reuse, and Part 9 reuses Part 8's client, so a part run on its own can fail with a `NameError`. You can stop after any part. Cost: a few cents of embeddings and model calls.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.py").exists())))
import course

course.setup()

## Part 1 · Measure retrieval before you change it

The lab's advice was "find the failing stage before you change anything". That needs a retrieval metric that does not involve the model at all. Two standard ones:

- **Recall@k**: the share of questions whose right passage is anywhere in the top *k*.
- **MRR** (mean reciprocal rank): the average of 1 / (rank of the first right passage). 1.0 means always first.

Build the same index as the service (one chunk per handbook section, `text-embedding-3-small`) and score it on the public Week 6 questions, which list the page (`doc_ids`) that answers each one.

In [ ]:
import json
import math
import os

from app.rag import load_chunks

client = course.openai_client()
MODEL = os.environ["FOUNDRY_MODEL"]
EMBEDDING_MODEL = os.environ["FOUNDRY_EMBEDDING_MODEL"]


def embed(texts: list[str], dimensions: int | None = None) -> list[list[float]]:
    extra = {"dimensions": dimensions} if dimensions else {}
    return [d.embedding for d in client.embeddings.create(model=EMBEDDING_MODEL, input=texts, **extra).data]


def dot(a: list[float], b: list[float]) -> float:
    return sum(x * y for x, y in zip(a, b, strict=True))  # the vectors are normalized, so dot = cosine


chunks = load_chunks()
chunk_vectors = embed([c.render() for c in chunks])
query_vectors: dict[str, list[float]] = {}


def add_queries(questions: list[str]) -> None:
    """Embed new questions in one batch and cache them."""
    new = [q for q in questions if q not in query_vectors]
    if new:
        query_vectors.update(zip(new, embed(new), strict=True))


def vector_search(question: str) -> list[int]:
    """Chunk indexes, best first."""
    scores = [dot(query_vectors[question], v) for v in chunk_vectors]
    return sorted(range(len(chunks)), key=scores.__getitem__, reverse=True)


def page(i: int) -> str:
    return chunks[i].doc_id


def section(i: int) -> tuple[str, str]:
    return (chunks[i].doc_id, chunks[i].heading)


def evaluate(search, queries: list[dict], label, k: int = 3) -> dict:
    """Recall@k and MRR. `label` turns a result index into what `relevant` contains."""
    hits, rr = 0, 0.0
    for q in queries:
        labels = [label(i) for i in search(q["question"])]
        hits += any(lab in q["relevant"] for lab in labels[:k])
        first = next((pos for pos, lab in enumerate(labels, 1) if lab in q["relevant"]), None)
        rr += 1 / first if first else 0.0
    return {f"recall@{k}": round(hits / len(queries), 2), "MRR": round(rr / len(queries), 2)}


rows = [json.loads(line) for line in Path("harness/data/week06.jsonl").read_text().splitlines() if line.strip()]
public = [{"question": r["question"], "relevant": set(r["doc_ids"])} for r in rows if r.get("doc_ids")]
add_queries([q["question"] for q in public])
print(f"{len(chunks)} chunks; {len(public)} public questions with a known answering page")
print("public questions, right page:", evaluate(vector_search, public, page, k=1))

Perfect, and that is a problem: **a metric stuck at 1.0 cannot tell you whether a change helps.** The public questions reuse the handbook's own words and only ask for the right page. Real riders do not write like the handbook.

So write a harder set: colloquial paraphrases, plus a few exact tokens (a station id, a phone number, a fee), each labeled with the **section** that answers it. Twenty labeled questions take fifteen minutes to write and are the most valuable retrieval work you will do.

In [ ]:
HARD = [  # (question, doc_id, section heading that answers it)
    ("Can my kid sit on the back of my bike?", "ride-rules", "Who can ride"),
    ("There's a thunderstorm rolling in. Can I still grab a bike?", "weather-and-closures", "Weather pause"),
    ("I left the bike leaning next to the rack. Am I still being billed?", "ride-rules", "Ending a ride"),
    ("How far will a fully charged electric bike take me?", "e-bikes", "Battery"),
    ("Someone on the phone asked for my password. Was that you?", "accounts-and-privacy", "What Support will never ask for"),
    ("Do leftover credits ever disappear from my account?", "refunds-and-credits", "Ride credits"),
    ("Do you have tricycles or hand cycles?", "accessibility", "Adaptive Ride program"),
    ("If I ride 12 minutes and 5 seconds, what do you bill?", "plans-and-pricing", "How minutes are counted"),
    ("Who signs off on a $40 goodwill credit?", "refunds-and-credits", "Who can approve what"),
    ("The bike got nicked while I was in a shop. How do I avoid the big fee?", "damage-and-lost-bikes", "Lost or stolen bikes"),
    ("How fast am I allowed to go along the lake?", "ride-rules", "Speed"),
    ("I fell and hurt my wrist. What now?", "safety-and-support", "If you are hurt"),
    ("The dock won't click my bike in.", "stations", "Broken docks"),
    ("Can I get my money back for the annual plan I bought yesterday?", "refunds-and-credits", "Refunds"),
    ("Is LS-03 open in January?", "stations", "Opening hours"),
    ("Is (555) 010-0100 really your number?", "safety-and-support", "How to reach Support"),
    ("What is the $2,500 fee?", "damage-and-lost-bikes", "Lost or stolen bikes"),
    ("What is special about LS-12?", "accessibility", "Adaptive Ride program"),
]
hard = [{"question": q, "relevant": {(doc, heading)}} for q, doc, heading in HARD]
assert all({(d, h)} <= {section(i) for i in range(len(chunks))} for _, d, h in HARD), "a label has a typo"
add_queries([q["question"] for q in hard])

print("hard questions, right section:", evaluate(vector_search, hard, section, k=1), evaluate(vector_search, hard, section, k=3))
for q in hard:
    top = [section(i) for i in vector_search(q["question"])[:3]]
    if not q["relevant"] & set(top):
        print(f"MISS  {q['question']!r}\n      expected {q['relevant']}, top 3 {top}")

Now the metric has room to move. Read the misses: they are **vocabulary mismatches**. The rider says "thunderstorm", the handbook says "lightning"; the rider says "leaning next to the rack", the handbook says "docked". Write these numbers down: Parts 2–6 are all compared against them.

## Part 2 · Keyword search from scratch: BM25

BM25 scores a chunk by the query terms it contains, weighted by how **rare** each term is in the collection (inverse document frequency), with diminishing returns for repeated terms ($k_1$) and a penalty for long chunks ($b$):

$$\text{score}(q, d) = \sum_{t \in q} \text{IDF}(t)\cdot\frac{f(t,d)\,(k_1+1)}{f(t,d) + k_1\left(1 - b + b\,\frac{|d|}{\text{avgdl}}\right)}$$

Twenty lines of Python:

In [ ]:
import re
from collections import Counter


def tokens(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


docs_tokens = [tokens(c.render()) for c in chunks]
avgdl = sum(map(len, docs_tokens)) / len(docs_tokens)
df = Counter(t for doc in docs_tokens for t in set(doc))
N = len(docs_tokens)


def bm25(query: str, k1: float = 1.5, b: float = 0.75) -> list[float]:
    scores = []
    for doc in docs_tokens:
        tf, score = Counter(doc), 0.0
        for t in tokens(query):
            if t in tf:
                idf = math.log(1 + (N - df[t] + 0.5) / (df[t] + 0.5))
                score += idf * tf[t] * (k1 + 1) / (tf[t] + k1 * (1 - b + b * len(doc) / avgdl))
        scores.append(score)
    return scores


def keyword_search(question: str) -> list[int]:
    scores = bm25(question)
    return sorted(range(len(chunks)), key=scores.__getitem__, reverse=True)


for name, search in [("vector", vector_search), ("keyword", keyword_search)]:
    print(f"{name:8s} public {evaluate(search, public, page, k=1)}   hard {evaluate(search, hard, section, k=1)}")

Keyword search knows nothing about meaning. It does well on the public questions, which reuse handbook words, and poorly on paraphrases. But not everywhere. Compare the rank of the right section, question by question:

In [ ]:
def rank_of_answer(search, q: dict) -> int | None:
    labels = [section(i) for i in search(q["question"])]
    return next((pos for pos, lab in enumerate(labels, 1) if lab in q["relevant"]), None)


print("vector  keyword  question")
for q in hard:
    print(f"{rank_of_answer(vector_search, q)!s:>6}  {rank_of_answer(keyword_search, q)!s:>7}  {q['question']}")

Vectors win on paraphrase ("fully charged" for "full battery", "kid" for "children"). Keyword search wins on rare exact tokens the handbook also contains, such as a fee amount or a station id, which an embedding blurs into "some number". **They fail on different questions.** That is the whole case for fusing them.

## Part 3 · Hybrid search with Reciprocal Rank Fusion

You cannot add a BM25 score (unbounded) to a cosine (between -1 and 1). **RRF** ignores scores and fuses **ranks**: each result list gives every item $w / (k + \text{rank})$, with $k = 60$ by convention and a weight $w$ per list. Azure AI Search fuses its keyword and vector results the same way.

In [ ]:
def rrf(rankings: list[tuple[list[int], float]], k: int = 60) -> list[int]:
    """Weighted Reciprocal Rank Fusion: every list adds weight / (k + rank) to each item."""
    fused: dict[int, float] = {}
    for ranking, weight in rankings:
        for rank, idx in enumerate(ranking, 1):
            fused[idx] = fused.get(idx, 0.0) + weight / (k + rank)
    return sorted(fused, key=fused.get, reverse=True)


def hybrid_search(question: str, keyword_weight: float = 1.0) -> list[int]:
    return rrf([(vector_search(question), 1.0), (keyword_search(question), keyword_weight)])


for name, search in [("vector", vector_search), ("keyword", keyword_search), ("hybrid", hybrid_search)]:
    print(f"{name:8s} {evaluate(search, hard, section, k=1)}  {evaluate(search, hard, section, k=3)}")

Surprise: equal-weight hybrid can rank the right section first *less* often than vectors alone. On paraphrased questions BM25's top results are mostly noise, and equal-weight RRF gives that noise the same vote. Fusion has knobs. Lower the keyword weight and watch both question sets:

In [ ]:
for weight in [1.0, 0.5, 0.3, 0.1]:
    search = lambda question, w=weight: hybrid_search(question, keyword_weight=w)  # noqa: E731
    print(f"keyword weight {weight}:  hard {evaluate(search, hard, section, k=1)}   public {evaluate(search, public, page, k=1)}")

A lighter keyword vote keeps BM25's wins on exact tokens without letting it overrule vectors on paraphrases. Azure AI Search exposes the same knob: a `weight` on each vector query in a hybrid request.

**Careful.** You just tuned a parameter on 18 questions and read the score on the same 18. That is overfitting in miniature: in a real project you tune on one set and report on another, and here every difference is one or two questions.

**Try it:** put weighted hybrid search into `HandbookIndex.search` in `app/rag.py` (the lab's *Go further*) and rerun the Week 6 harness. Did `w06.ask.citations` move?

## Part 4 · Query rewriting and multi-query fusion

The thunderstorm question fails because the handbook says "lightning". A model knows those are related. Ask it to rewrite each question into handbook vocabulary, then search with the rewrite. Planning queries, running them and fusing the results is the core move of **agentic retrieval** (lecture §8 and §12).

In [ ]:
def rewrite(question: str) -> str:
    response = client.responses.create(
        model=MODEL,
        reasoning={"effort": "none"},
        instructions="Rewrite the rider's question as a short search query in the vocabulary of a "
        "bike-share policy handbook. Return only the query.",
        input=question,
    )
    return response.output_text.strip()


rewrites = {q["question"]: rewrite(q["question"]) for q in hard}
add_queries(list(rewrites.values()))


def rewrite_search(question: str) -> list[int]:
    return vector_search(rewrites[question])


def multi_query_search(question: str) -> list[int]:
    """Original question + rewrite + a light keyword vote, fused with RRF."""
    return rrf([(vector_search(question), 1.0), (rewrite_search(question), 1.0), (keyword_search(question), 0.3)])


for name, search in [("original", vector_search), ("rewrite only", rewrite_search), ("original + rewrite + keyword", multi_query_search)]:
    print(f"{name:30s} {evaluate(search, hard, section, k=1)}")
print()
for q in hard:
    original, rewritten = rank_of_answer(vector_search, q), rank_of_answer(rewrite_search, q)
    if original != rewritten:
        print(f"rank {original} -> {rewritten}  {q['question']!r}\n             rewrite: {rewrites[q['question']]!r}")

The rewrite alone fixes some questions and breaks others: read the rewrites above and you will see it drop words that mattered. Fused with the original, it keeps both chances. Two lessons:

1. A model inside the retrieval loop is one more stage. Measure it like any other; it is not automatically an improvement.
2. **Fusion is what makes multiple queries safe.** Each query only adds votes; no single bad query decides.

Rewrites are sampled, so your numbers may differ by a question or two from your neighbor's. Managed services run this loop at scale, with query planning, parallel subqueries, semantic reranking and permissions: read [agentic retrieval in Azure AI Search](https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-overview).

## Part 5 · Chunking: sections versus fixed-size windows

The service chunks by handbook section. The most common alternative is a fixed window of words with overlap. Windows have no section label, so compare both at the page level on the hard questions:

In [ ]:
def windows(text: str, size: int = 40, overlap: int = 10) -> list[str]:
    words = text.split()
    step = size - overlap
    return [" ".join(words[i : i + size]) for i in range(0, max(len(words) - overlap, 1), step)]


pages: dict[str, str] = {}
for c in chunks:
    pages[c.doc_id] = pages.get(c.doc_id, "") + f"\n{c.heading}. {c.text}"

window_chunks = [(doc_id, w) for doc_id, text in pages.items() for w in windows(text)]
window_vectors = embed([w for _, w in window_chunks])
hard_pages = [{"question": q["question"], "relevant": {doc for doc, _ in q["relevant"]}} for q in hard]


def window_search(question: str) -> list[int]:
    scores = [dot(query_vectors[question], v) for v in window_vectors]
    return sorted(range(len(window_chunks)), key=scores.__getitem__, reverse=True)


print(f"{len(chunks)} sections versus {len(window_chunks)} windows")
print("sections:", evaluate(vector_search, hard_pages, page, k=1))
print("windows: ", evaluate(window_search, hard_pages, lambda i: window_chunks[i][0], k=1))

Windows may well find the right page more often: smaller, overlapping pieces give a question more chances to match. Now look at what the model would actually receive:

In [ ]:
for question in ["The bike got nicked while I was in a shop. How do I avoid the big fee?", "Is LS-03 open in January?"]:
    print(question)
    print("  SECTION:", chunks[vector_search(question)[0]].render().replace("\n", " | "))
    print("  WINDOW: ", window_chunks[window_search(question)[0]][1])
    print()

A window starts and stops wherever the word count says. It can separate a rule from its exception (the $150 police-report fee from the $1,200 lost-bike fee), and it carries no heading to cite. A better retrieval score is not automatically a better answer, which is why the lab's grader checks facts and citations, not retrieval alone. Structure-aware chunks keep meaning together when the source has structure. Try `windows(text, size=80, overlap=20)` and watch both effects.

## Part 6 · Embedding size: 1,536, 256 and 64 dimensions

`text-embedding-3` models accept a `dimensions` parameter that returns shorter vectors. Shorter vectors cost less to store and compare. What do they cost in quality, here?

In [ ]:
hard_questions = [q["question"] for q in hard]

for dims in [1536, 256, 64]:
    small_chunks = embed([c.render() for c in chunks], dimensions=dims)
    small_queries = dict(zip(hard_questions, embed(hard_questions, dimensions=dims), strict=True))

    def small_search(question: str, small_chunks=small_chunks, small_queries=small_queries) -> list[int]:
        scores = [dot(small_queries[question], v) for v in small_chunks]
        return sorted(range(len(chunks)), key=scores.__getitem__, reverse=True)

    print(f"{dims:5d} dims, {dims * 4:5d} bytes per vector:  {evaluate(small_search, hard, section, k=1)}  {evaluate(small_search, hard, section, k=3)}")

Quality degrades gradually, not in proportion to size. These models are trained so the first dimensions carry the most information (Matryoshka representation learning), which is why cutting the vector works at all. At millions of chunks, six times less storage is real money and latency; at 35 chunks it is irrelevant. Measure on your own data before you choose.

## Part 7 · The tool-calling protocol at its edges

Three controls from §9, observed directly on the wire. First, **parallel tool calls**: one question that needs two independent lookups.

In [ ]:
from app import domain

TOOLS = [
    {"type": "function", "name": "quote_fare", "strict": True,
     "description": "Exact ride fee in US dollars for one ride. Always use this instead of computing prices.",
     "parameters": {"type": "object", "additionalProperties": False, "required": ["plan", "minutes", "ebike"],
                    "properties": {"plan": {"type": "string", "enum": ["single", "day_pass", "member", "student"]},
                                   "minutes": {"type": "integer"}, "ebike": {"type": "boolean"}}}},
    {"type": "function", "name": "get_station", "strict": True,
     "description": "Live classic bikes, e-bikes and free docks at one station.",
     "parameters": {"type": "object", "additionalProperties": False, "required": ["station_id"],
                    "properties": {"station_id": {"type": "string"}}}},
]
QUESTION = "I'm a member. What does a 50-minute classic ride cost, and are there e-bikes at LS-08?"

for parallel in [True, False]:
    r = client.responses.create(model=MODEL, input=QUESTION, tools=TOOLS, parallel_tool_calls=parallel)
    calls = [(i.name, i.arguments) for i in r.output if i.type == "function_call"]
    print(f"parallel_tool_calls={parallel!s:5}: {len(calls)} call(s) in the first response: {calls}")

With parallel calls the model asks for both lookups at once: one round trip instead of two, so lower latency. Agent Framework runs parallel calls concurrently for you.

Second, **`tool_choice`**. `"required"` forces a tool call even when none is needed; `"none"` forbids tools. Useful for a step that must look something up, or one that must not act:

In [ ]:
for choice in ["auto", "required", "none"]:
    r = client.responses.create(model=MODEL, input="Hi! What can you help me with?", tools=TOOLS, tool_choice=choice)
    calls = [i.name for i in r.output if i.type == "function_call"]
    print(f"tool_choice={choice:9s} calls={calls}  text={r.output_text[:60]!r}")

Third, **errors as data**. A tool that raises crashes your loop; a tool that returns an explanation lets the model recover. `domain.get_station` returns an error object for an unknown id; send it back and see what the model does with it:

In [ ]:
r = client.responses.create(model=MODEL, input="How many docks are free at station LS-99?", tools=TOOLS)
call = next(i for i in r.output if i.type == "function_call")
result = domain.get_station(json.loads(call.arguments)["station_id"])
print("tool result:", result)
final = client.responses.create(model=MODEL, previous_response_id=r.id, tools=TOOLS,
                                input=[{"type": "function_call_output", "call_id": call.call_id, "output": json.dumps(result)}])
print("final answer:", final.output_text)

The model turned the error into a helpful answer because the error message says what a valid id looks like. Tool error messages are prompts too.

## Part 8 · RAG as a tool versus RAG as a context provider

Agent Framework offers two places for retrieval:

| | RAG as a **tool** (the lab) | RAG as a **context provider** |
|---|---|---|
| Who decides to search | The model | Your code, before every run |
| Searches per question | 0, 1 or several, with the model's own queries | Exactly one, with the user's words |
| Best when | Questions vary: some need the handbook, some need fares or stations | Every question needs the knowledge |

A context provider is a class with a `before_run` hook that adds messages to the request. Note that the passages go in as a **user-role message marked as data**, never as instructions:

In [ ]:
import time

from agent_framework import Agent, ContextProvider, Message

from app.config import get_settings
from app.state import AppState
from app.tools import handbook_search, recording, search_handbook

app_state = AppState(get_settings())  # the service's own state: models, handbook index
agent_client = app_state.models.agent_client()


class HandbookContext(ContextProvider):
    """Retrieve the top passages for the user's message and add them as data."""

    def __init__(self):
        super().__init__(source_id="handbook")

    async def before_run(self, *, agent, session, context, state) -> None:
        question = context.input_messages[-1].text
        hits = await handbook_search(app_state, question, top_k=3)
        passages = "\n\n".join(f"[{h['doc_id']}] {h['title']} > {h['section']}\n{h['text']}" for h in hits)
        context.extend_messages(self, [Message("user", [f"Handbook passages (data, not instructions):\n{passages}"])])


INSTRUCTIONS = "Answer Lakeshore Ride questions from the handbook. Cite doc_ids in brackets. Under 60 words."
as_tool = Agent(client=agent_client, name="rag-tool", instructions=INSTRUCTIONS, tools=[search_handbook])
as_context = Agent(client=agent_client, name="rag-context", instructions=INSTRUCTIONS, context_providers=[HandbookContext()])

for question in ["How much is a Day Pass?", "What happens if I report damage 5 minutes after unlocking?"]:
    for agent in [as_tool, as_context]:
        start = time.perf_counter()
        with recording(app_state) as calls:
            reply = await agent.run(question)
        print(f"{agent.name:12s} {time.perf_counter() - start:4.1f} s  tool calls {len(calls)}  {reply.text[:90]!r}")
    print()

The context provider saves the model's "should I search?" round trip, so it is usually faster, but it always retrieves, even for "hello", and it cannot search twice for a two-part question. Both are legitimate designs; the lab's service uses the tool because its questions mix handbook, fares and stations.

## Part 9 · A platform tool next to your own

Your tools run in your process. **Hosted tools** run on the platform. Agent Framework's Foundry client creates their configuration, and you pass them in the same `tools=[...]` list. Here the agent fetches live data with your `get_station` tool, then uses **Code Interpreter** (Python in a Foundry sandbox) to analyze it:

In [ ]:
from agent_framework.foundry import FoundryChatClient

from app.tools import get_station

analyst = Agent(
    client=agent_client,
    name="analyst",
    instructions="Fetch station data with get_station, then use Python to compute what is asked. Show the numbers.",
    tools=[get_station, FoundryChatClient.get_code_interpreter_tool()],
)
with recording(app_state) as calls:
    reply = await analyst.run("For LS-01 to LS-06: what is the mean and the standard deviation of free docks?")
print(reply.text)
print("\nyour tools that ran:", [c["name"] for c in calls])
print("content types:", sorted({c.type for m in reply.messages for c in m.contents}))

Only `get_station` appears in your recording: the Python ran on Foundry, not on your machine. That is the trade-off from §11: less code for you, but execution, data and logs move to the platform.

**Managed RAG.** The next step after Parts 1–6 is not more hand-written search; it is a search service. Read [What is Foundry IQ?](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/what-is-foundry-iq) and [agentic retrieval](https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-overview): the same ideas (hybrid, reranking, query planning, citations) plus permissions and many sources, run for you.

### What to remember

- Measure retrieval **without the model**, on questions **hard enough to move**: a metric stuck at 1.0 guides nothing.
- **BM25** and **vectors** fail on different questions; **RRF** fuses ranks, and its weights matter. Tune on one set, report on another.
- **Query rewriting** helps only when **fused** with the original: that is the seed of agentic retrieval.
- Chunk by **structure** when you have it; a better retrieval score from windows can still hand the model a worse passage.
- Embedding **dimensions** are a cost knob; measure the quality you give up.
- Parallel calls cut latency; `tool_choice` constrains a step; **errors are data** the model can use.
- In Agent Framework, retrieval can be a **tool** (model decides) or a **context provider** (code decides); hosted tools sit next to yours.

**Learn more:** [Hybrid search](https://learn.microsoft.com/en-us/azure/search/hybrid-search-overview) · [Semantic ranking](https://learn.microsoft.com/en-us/azure/search/semantic-search-overview) · [Function calling (OpenAI)](https://developers.openai.com/api/docs/guides/function-calling) · [Agent Framework overview](https://learn.microsoft.com/en-us/agent-framework/overview/)